# Actividad 4 | Aplicar algoritmos de aprendizaje no supervisado con PySpark

**Instituto Tecnológico y de Estudios Superiores de Monterrey**  
**Maestría en Inteligencia Artificial Aplicada**  
**Análisis de grandes volúmenes de datos**  
**Docente:** Dr. Iván Olmos Pineda

* Palma Palacios, Christian Ricardo — A01686081

**Fecha:** 30 de mayo de 2026

## 1. Introducción Teórica

El aprendizaje no supervisado es una rama del Machine Learning en la cual los algoritmos trabajan con datos que no poseen etiquetas o clases definidas. A diferencia del aprendizaje supervisado, donde el modelo aprende a partir de ejemplos etiquetados, en el aprendizaje no supervisado el objetivo es descubrir patrones, estructuras, relaciones o agrupaciones ocultas dentro de los datos.

### 1.1 Principales técnicas de aprendizaje no supervisado

**a) Clustering (Agrupamiento)**
El objetivo es agrupar observaciones similares en conjuntos llamados clusters. Algoritmos representativos:
- K-Means. Es uno de los algoritmos más utilizados. Divide los datos en k grupos minimizando la distancia entre cada punto y el centroide de su grupo.
- Gaussian Mixture Model (GMM). Asume que los datos provienen de una combinación de distribuciones gaussianas.
- Hierarchical Clustering. Construye una jerarquía de agrupamientos.
- DBSCAN. Agrupa puntos según densidad.

**b) Reducción de dimensionalidad**
Busca representar datos de alta dimensión en menos variables conservando la mayor información posible. Algoritmos representativos:
- PCA (Principal Component Analysis). Transforma variables correlacionadas en componentes principales independientes.
- SVD (Singular Value Decomposition). Muy utilizado en sistemas de recomendación y procesamiento de texto.

**c) Reglas de asociación**
Permiten descubrir relaciones frecuentes entre elementos de un conjunto de datos. Algoritmos representativos:
- Apriori. Encuentra patrones frecuentes y reglas de asociación.
- FP-Growth. Versión optimizada de Apriori para grandes volúmenes de datos.

###1.2. Algoritmos no supervisados disponibles en PySpark
PySpark, a través de la librería MLlib, ofrece múltiples algoritmos de aprendizaje no supervisado diseñados para procesamiento distribuido de grandes volúmenes de datos.

Algoritmos de clustering disponibles:

| Algoritmo              | Clase PySpark                  | Descripción                                |
| ---------------------- | ------------------------------ | ------------------------------------------ |
| K-Means                | `pyspark.ml.clustering.KMeans` | Agrupamiento basado en centroides          |
| Bisecting K-Means      | `BisectingKMeans`              | Variante jerárquica de K-Means             |
| Gaussian Mixture Model | `GaussianMixture`              | Modelo probabilístico basado en gaussianas |
| LDA                    | `LatentDirichletAllocation`    | Modelado de tópicos                        |

Algoritmos de reducción de dimensionalidad:

| Algoritmo | Clase PySpark            |
| --------- | ------------------------ |
| PCA       | `pyspark.ml.feature.PCA` |

Algoritmos de reglas de asociación:

| Algoritmo | Clase PySpark             |
| --------- | ------------------------- |
| FP-Growth | `pyspark.ml.fpm.FPGrowth` |

###1.3. Algoritmos recomendados para el proyecto
Considerando que el dataset con el que trabajaremos es “Microsoft_GUIDE_Train.csv” con las siguientes caracteristicas:
- El objetivo principal del dataset es clasificar incidentes de ciberseguridad en categorías como: True Positive (ataque real), False Positive (falsa alarma), Benign Positive (evento legítimo, autorizado o inofensivo).
- Contiene datos de telemetría a múltiples niveles: evidencia, alertas e incidentes.
- Integra información de más de 6,100 organizaciones, lo que lo convierte en un dataset realista y diverso.
- Incluye millones de registros (más de 13 millones de evidencias) y eventos etiquetados.
- Está alineado con el framework MITRE ATT&CK, cubriendo cientos de técnicas de ataque.

Los dos algoritmos de aprendizaje no supervisado más recomendables en PySpark son:

**1. K-Means Clustering**

K-Means está altamente optimizado en Spark y funciona muy bien sobre: datasets distribuidos, millones de registros, procesamiento paralelo. Esto es crítico considerando las más de 13 millones de evidencias. Comparado con otros algoritmos: consume menos memoria, converge rápidamente, es eficiente en clusters grandes.

**2. Gaussian Mixture Model (GMM)**

En ciberseguridad los ataques no siempre forman grupos compactos, existen solapamientos entre clases, algunos eventos son ambiguos. GMM maneja esto mucho mejor que K-Means. En lugar de asignar rígidamente un registro a un cluster, GMM calcula probabilidades (70% cluster ataque). Comparado con K-Means es más lento, consume más memoria, escala peor.

##2. Selección de los datos

A continuación, se instalan las dependencias necesarias para ejecutar PySpark en el entorno de Google Colab.

In [1]:
# Instalación de Java y descarga de Spark
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar -xzf spark-3.5.8-bin-hadoop3.tgz
!pip install -q findspark

In [2]:
# Configuración de variables de entorno
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.8-bin-hadoop3"

In [3]:
# Inicialización de la sesión de Spark
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("BigData_Particionamiento") \
    .getOrCreate()

# Habilitar formato de salida mejorado para DataFrames
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

Se monta la unidad de Google Drive y se carga el archivo CSV en un DataFrame de PySpark.

In [4]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Definir ruta del archivo
file_path = "/content/drive/MyDrive/Colab Notebooks/MNA/BigData/ProyectoEtapa1/Microsoft_GUIDE_Train.csv"

Mounted at /content/drive


In [5]:
# Cargar el dataset
df = spark.read.csv(file_path, header=True, sep=",", inferSchema=True)

# Mostrar las primeras 2 filas para verificar la carga
df.limit(2).toPandas()

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,...,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
0,180388628218,0,612,123247,2024-06-04 06:05:15,7,6,InitialAccess,None,TruePositive,...,None,None,5,66,None,None,None,31,6,3
1,455266534868,88,326,210035,2024-06-14 03:01:25,58,43,Exfiltration,None,FalsePositive,...,None,None,5,66,None,None,None,242,1445,10630


**Preprocesamiento**

Asegurar la calidad de los datos antes del particionamiento (muestreo) y antes del aprendizaje supervisado.

In [6]:
from pyspark.sql.functions import col, when, count, lit

# Eliminar registros sin variable objetivo
df = df.filter(col("IncidentGrade").isNotNull())

# Eliminar duplicados
df = df.dropDuplicates()

# Eliminar columnas >90% NULL
cols_to_drop = [
    "ActionGrouped",
    "ActionGranular",
    "EmailClusterId",
    "ResourceType",
    "Roles",
    "AntispamDirection",
    "ThreatFamily"
]
df = df.drop(*cols_to_drop)

# Imputar variables moderadamente incompletas
fill_values = {
    "MitreTechniques": "Unknown",
    "SuspicionLevel": "Unknown",
    "LastVerdict": "Unknown"
}
df = df.fillna(fill_values)

# Extraer variables temporales
from pyspark.sql.functions import hour, dayofweek, month
df = df.withColumn("EventHour", hour(col("Timestamp")))
df = df.withColumn("EventDayOfWeek", dayofweek(col("Timestamp")))
df = df.withColumn("EventMonth", month(col("Timestamp")))

In [7]:
# Reducir cardinalidad (Top-15) para aprendizaje no supervisado
top_categories = [
    row["Category"]
    for row in df.groupBy("Category")
    .count()
    .orderBy(col("count").desc())
    .limit(15)
    .collect()
]

df = df.withColumn(
    "CategoryReduced",
    when(
        col("Category").isin(top_categories),
        col("Category")
    ).otherwise("OtherCategory")
)

In [8]:
# Obtener cantidad total de registros limpios
total_records = df.count()
print(f"Cantidad total de registros limpios: {total_records:,}")

Cantidad total de registros limpios: 9,442,956


**Muestreo**

Para evitar una explosión combinatoria y estratos demasiado pequeños, se aplica una estrategia de agrupación *Top-N* sobre las variables categóricas de mayor cardinalidad.

In [9]:
# Agrupación de la variable Category (Top 7)
top_categories = [
    "InitialAccess",
    "Exfiltration",
    "SuspiciousActivity",
    "CommandAndControl",
    "Impact",
    "CredentialAccess",
    "Execution"
]
df = df.withColumn(
    "CategoryGroup",
    when(col("Category").isin(top_categories), col("Category"))
    .otherwise("OtherCategory")
)

# Agrupación de la variable EntityType (Top 2)
df = df.withColumn(
    "EntityTypeGroup",
    when(col("EntityType") == "Ip", "Ip")
    .when(col("EntityType") == "User", "User")
    .otherwise("Other")
)

Se definen las columnas que conformarán las reglas de particionamiento.

In [10]:
# Definir columnas de particionamiento
partition_columns = [
    "IncidentGrade",
    "CategoryGroup",
    "EntityTypeGroup",
    "EvidenceRole"
]

Muestreo estratificado proporcional sin reemplazo, utilizando como estratos las combinaciones observadas empíricamente de las variables IncidentGrade, CategoryGroup, EntityTypeGroup y EvidenceRole

In [11]:
from pyspark.sql.functions import concat_ws

# =====================================================
# 1. Crear columna estrato (partición)
# =====================================================

df = df.withColumn(
    "Stratum",
    concat_ws(
        "_",
        col("IncidentGrade"),
        col("CategoryGroup"),
        col("EntityTypeGroup"),
        col("EvidenceRole")
    )
)

# =====================================================
# 2. Obtener lista de estratos observados
# =====================================================

strata_df = df.select("Stratum").distinct()

strata_list = [row["Stratum"] for row in strata_df.collect()]

print(f"Número de estratos observados: {len(strata_list)}")

# =====================================================
# 3. Definir fracción de muestreo por estrato
# =====================================================

sample_fraction = 0.10

fractions = {
    stratum: sample_fraction
    for stratum in strata_list
}

# =====================================================
# 4. Extraer muestra estratificada M
# =====================================================

seed_value = 42

sample_M = df.sampleBy(
    "Stratum",
    fractions=fractions,
    seed=seed_value
)

# =====================================================
# 5. Persistir M en memoria
# =====================================================

# Cache en memoria RAM y Disco
sample_M = sample_M.cache()
# Materializar cache UNA sola vez
_ = sample_M.count()

# =====================================================
# 6. Verificar tamaño de muestra
# =====================================================

sample_count = sample_M.count()

print(f"Total registros dataset original: {total_records:,}")
print(f"Total registros muestra M: {sample_count:,}")
print(f"Porcentaje real: {(sample_count/total_records)*100:.4f}%")

Número de estratos observados: 115
Total registros dataset original: 9,442,956
Total registros muestra M: 944,874
Porcentaje real: 10.0061%


Verificación de representatividad. La muestra mantiene aproximadamente las proporciones originales.

In [12]:
# Distribución original
original_distribution = df.groupBy(*partition_columns) \
    .count() \
    .withColumnRenamed("count", "OriginalCount")

# Distribución de la muestra
sample_distribution = sample_M.groupBy(*partition_columns) \
    .count() \
    .withColumnRenamed("count", "SampleCount")

# Comparación
comparison = original_distribution.join(
    sample_distribution,
    on=partition_columns,
    how="left"
)

comparison = comparison.fillna(0)
comparison = comparison.withColumn(
    "SamplePct",
    (col("SampleCount") / col("OriginalCount")) * 100
)
comparison.orderBy(col("OriginalCount").desc()).show(3, truncate=False)

+--------------+-------------+---------------+------------+-------------+-----------+------------------+
|IncidentGrade |CategoryGroup|EntityTypeGroup|EvidenceRole|OriginalCount|SampleCount|SamplePct         |
+--------------+-------------+---------------+------------+-------------+-----------+------------------+
|TruePositive  |InitialAccess|Other          |Related     |920775       |92327      |10.027096739159946|
|BenignPositive|Exfiltration |Other          |Impacted    |823594       |82394      |10.004201099085229|
|TruePositive  |InitialAccess|Ip             |Related     |602878       |60187      |9.98328019931064  |
+--------------+-------------+---------------+------------+-------------+-----------+------------------+
only showing top 3 rows



**Submuestreo**

Construcción de una muestra M' para evitar que los tiempos de procesamiento sean altos. M' será un subconjunto de M.
Se aplicará nuevamente el muestreo estratificado proporcional sin reemplazo
utilizando los mismos estratos definidos previamente.

In [13]:
# Definir fracción de reducción
sample_fraction_M_prime = 0.003

# Verificar tamaño actual de M
M_count = sample_count

# =====================================================
# Definir fracciones por estrato
# =====================================================

fractions_M_prime = {
    stratum: sample_fraction_M_prime
    for stratum in strata_list
}

# =====================================================
# Extraer muestra M'
# =====================================================

seed_value = 42

M_prime = sample_M.sampleBy(
    "Stratum",
    fractions=fractions_M_prime,
    seed=seed_value
)

# =====================================================
# Persistir M' en memoria
# =====================================================

# Cache en memoria RAM y Disco
M_prime = M_prime.cache()
# Materializar cache UNA sola vez
_ = M_prime.count()

# =====================================================
# Verificar tamaño
# =====================================================

M_prime_count = M_prime.count()

print(f"Registros en M: {M_count:,}")
print(f"Registros en M': {M_prime_count:,}")
print(f"Porcentaje respecto a M: {(M_prime_count/M_count)*100:.4f}%")

Registros en M: 944,874
Registros en M': 2,821
Porcentaje respecto a M: 0.2986%


Verificación de representatividad de M'

In [14]:
# Distribución en M'
dist_M_prime = M_prime.groupBy(*partition_columns) \
    .count() \
    .withColumnRenamed("count", "Count_M_prime")

# Distribución en M
dist_M = sample_M.groupBy(*partition_columns) \
    .count() \
    .withColumnRenamed("count", "Count_M")

# Comparación
comparison_M = dist_M.join(
    dist_M_prime,
    on=partition_columns,
    how="left"
)
comparison_M = comparison_M.fillna(0)
comparison_M = comparison_M.withColumn(
    "Pct_M_prime",
    (col("Count_M_prime") / col("Count_M")) * 100
)
comparison_M.orderBy(col("Count_M").desc()).show(3, truncate=False)

+--------------+-------------+---------------+------------+-------+-------------+-------------------+
|IncidentGrade |CategoryGroup|EntityTypeGroup|EvidenceRole|Count_M|Count_M_prime|Pct_M_prime        |
+--------------+-------------+---------------+------------+-------+-------------+-------------------+
|TruePositive  |InitialAccess|Other          |Related     |92327  |257          |0.2783584433589308 |
|BenignPositive|Exfiltration |Other          |Impacted    |82394  |250          |0.3034201519528121 |
|TruePositive  |InitialAccess|Ip             |Related     |60187  |153          |0.25420771927492647|
+--------------+-------------+---------------+------------+-------+-------------+-------------------+
only showing top 3 rows



##3. Preparación del conjunto de entrenamiento, validación y prueba

Una vez construida la muestra reducida M', el siguiente paso consiste en dividirla en subconjuntos de:

- Entrenamiento (Training Set)
- Validación (Validation Set)
- Prueba (Test Set)

En aprendizaje supervisado preservar clases explícitas es fundamental, pero en aprendizaje no supervisado:
- No entrenamos usando la variable objetivo ("IncidentGrade ").
- Buscamos estructuras latentes.
- Dependemos fuertemente de la distribución geométrica real de los datos.

El split aleatorio por filas puede distorsionar clusters raros. Además el split aleatorio puede generar Leakage estructural:
- El mismo dispositivo ("DeviceId") puede aparecer en TRAIN y TEST.
- La misma organización ("OrgId") puede aparecer en TRAIN y VALIDATION.
- El mismo incidente puede fragmentarse entre subconjuntos.

**Técnica recomendada**

Group-Based Splitting (división por grupos) usando una entidad estructural como "OrgId", porque el dataset integra más de 6,100 organizaciones. Al dividir por OrgId:
- TRAIN contiene organizaciones distintas de TEST.
- El modelo realmente generaliza y se evita el leakage operativo.


La división recomendada es:

| Conjunto      | Porcentaje | Objetivo                  |
| ------------- | ---------- | ------------------------- |
| Entrenamiento | 70%        | Aprendizaje del modelo    |
| Validación    | 15%        | Ajuste de hiperparámetros |
| Prueba        | 15%        | Evaluación final          |


In [15]:
# =====================================================
# División Train / Validation / Test
# =====================================================

# División por OrgId
# 70% entrenamiento
# 15% validación
# 15% prueba
orgs = M_prime.select("OrgId").distinct()

train_orgs, val_orgs, test_orgs = orgs.randomSplit(
    [0.70, 0.15, 0.15],
    seed=42
)

train_df = M_prime.join(train_orgs, on="OrgId")
val_df = M_prime.join(val_orgs, on="OrgId")
test_df = M_prime.join(test_orgs, on="OrgId")

# Persistir subconjuntos
train_df = train_df.cache()
val_df = val_df.cache()
test_df = test_df.cache()

# Verificación eficiente
print({
    "TRAIN": train_df.count(),
    "VALIDATION": val_df.count(),
    "TEST": test_df.count()
})

{'TRAIN': 1926, 'VALIDATION': 583, 'TEST': 312}


##4. Construcción de modelos de aprendizaje no supervisado

Para este proyecto de ciberseguridad y clustering no supervisado en PySpark, la mejor métrica para evaluar los modelos es el Silhouette Score, porque:

- Mide cohesión y separación entre clusters.
- No requiere etiquetas reales.
- Está implementada directamente en PySpark.
- Es ampliamente utilizada en clustering de anomalías y telemetría.
- Funciona correctamente tanto para K-Means como para GMM.

El valor del Silhouette Score se encuentra entre:

- Cercano a 1 → clusters bien definidos.
- Cercano a 0 → clusters superpuestos.
- Negativo → clustering deficiente.

###4.1. Selección de variables predictoras

Se utilizarán variables relevantes para clustering de comportamiento en ciberseguridad.

In [16]:
# ==========================================
# Variables predictoras
# ==========================================

categorical_cols = [
    "CategoryReduced",
    "MitreTechniques",
    "EntityType",
    "EvidenceRole",
    "SuspicionLevel",
    "LastVerdict"
]

numeric_cols = [
    "EventHour",
    "EventDayOfWeek",
    "EventMonth"
]

###4.2 Transformación de variables (Frequency Encoding)

In [17]:
from pyspark.sql.functions import count, col

# ==========================================
# Frequency Encoding
# ==========================================

for c in categorical_cols:

    freq_df = train_df.groupBy(c).agg(
        count("*").alias(f"{c}_freq")
    )

    # TRAIN
    train_df = train_df.join(freq_df, on=c, how="left")

    # VALIDATION
    val_df = val_df.join(freq_df, on=c, how="left")

    # TEST
    test_df = test_df.join(freq_df, on=c, how="left")

###4.3. Construcción del vector de características

In [18]:
from pyspark.ml.feature import VectorAssembler

# ==========================================
# Variables finales
# ==========================================

feature_cols = [
    f"{c}_freq" for c in categorical_cols
] + numeric_cols

# ==========================================
# Frequency columns
# ==========================================

freq_cols = [f"{c}_freq" for c in categorical_cols]

# ==========================================
# Imputar NULLs
# ==========================================

fill_freq = {
    c: 0 for c in freq_cols
}

train_df = train_df.fillna(fill_freq)
val_df = val_df.fillna(fill_freq)
test_df = test_df.fillna(fill_freq)

# ==========================================
# Convertir a double
# ==========================================

for c in feature_cols:
    train_df = train_df.withColumn(c, col(c).cast("double"))
    val_df = val_df.withColumn(c, col(c).cast("double"))
    test_df = test_df.withColumn(c, col(c).cast("double"))

# ==========================================
# VectorAssembler
# ==========================================

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

train_df = assembler.transform(train_df)
val_df = assembler.transform(val_df)
test_df = assembler.transform(test_df)

# =====================================================
# Verificación
# =====================================================

train_df.select("features").show(5, truncate=False)

+-----------------------------------------------------+
|features                                             |
+-----------------------------------------------------+
|[1016.0,348.0,154.0,1135.0,1582.0,1421.0,6.0,1.0,6.0]|
|[181.0,987.0,143.0,1135.0,344.0,323.0,22.0,3.0,6.0]  |
|[54.0,987.0,384.0,791.0,1582.0,1421.0,10.0,3.0,6.0]  |
|[1016.0,44.0,151.0,791.0,1582.0,1421.0,15.0,2.0,5.0] |
|[181.0,987.0,143.0,1135.0,344.0,323.0,5.0,3.0,6.0]   |
+-----------------------------------------------------+
only showing top 5 rows



###4.4. Escalamiento y PCA

In [19]:
from pyspark.ml.feature import StandardScaler

# ==========================================
# Escalamiento
# ==========================================

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withMean=False,
    withStd=True
)

scaler_model = scaler.fit(train_df)

train_df = scaler_model.transform(train_df)
val_df = scaler_model.transform(val_df)
test_df = scaler_model.transform(test_df)

from pyspark.ml.feature import PCA

# ==========================================
# PCA (significa una carga computacional alta)
# ==========================================

pca = PCA(
    k=5,
    inputCol="scaledFeatures",
    outputCol="pcaFeatures"
)

pca_model = pca.fit(train_df)

train_df = pca_model.transform(train_df)
val_df = pca_model.transform(val_df)
test_df = pca_model.transform(test_df)

# =====================================================
# Verificación
# =====================================================

train_df.select("features", "scaledFeatures", "pcaFeatures").show(5, truncate=False)

+-----------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|features                                             |scaledFeatures                                                                                                                                                            |pcaFeatures                                                                                        |
+-----------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|[1016.0,348.0,154.

###4.5 Entrenamiento y validación de modelos

In [26]:
# En datasets pequeños (M') se debe evitar que Spark siga trabajando distribuídamente
train_df = train_df.coalesce(1)
val_df = val_df.coalesce(1)
test_df = test_df.coalesce(1)

In [27]:
from pyspark.ml.evaluation import ClusteringEvaluator

# ==========================================
# Evaluador
# ==========================================

evaluator = ClusteringEvaluator(
    featuresCol="scaledFeatures",  #pca es muy costoso y no mejora las metricas
    predictionCol="prediction",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

# ==========================================
# Lista de K
# ==========================================

k_values = [5, 9]

K-Means

In [28]:
from pyspark.ml.clustering import KMeans

# ==========================================
# K-MEANS
# ==========================================

best_kmeans_model = None
best_kmeans_score = -1
best_k = None

for k in k_values:

    kmeans = KMeans(
        k=k,
        seed=42,
        featuresCol="scaledFeatures",
        predictionCol="prediction",
        maxIter=20  # mejora performance
    )

    model = kmeans.fit(train_df)

    # TRAIN
    train_pred = model.transform(train_df)

    train_silhouette = evaluator.evaluate(train_pred)

    # VALIDATION
    val_pred = model.transform(val_df)

    val_silhouette = evaluator.evaluate(val_pred)

    diff_pct = abs(train_silhouette - val_silhouette)

    print("=" * 50)
    print(f"KMEANS - K={k}")
    print(f"Train Silhouette: {train_silhouette:.4f}")
    print(f"Validation Silhouette: {val_silhouette:.4f}")
    print(f"Diferencia absoluta: {diff_pct:.4f}")

    # Verificación sobreentrenamiento
    if diff_pct < 0.03:
        print("El modelo NO está sobreentrenado.")
    else:
        print("Posible sobreentrenamiento.")

    # Verificación subentrenamiento (> 0.50 en telemetría SOC indica clusters relativamente bien definidos)
    if train_silhouette < 0.50 and val_silhouette < 0.50:
        print("Posible subentrenamiento.")
    else:
        print("No hay subentrenamiento.")

    # Mejor modelo
    if val_silhouette > best_kmeans_score:
        best_kmeans_score = val_silhouette
        best_kmeans_model = model
        best_k = k

KMEANS - K=5
Train Silhouette: 0.3401
Validation Silhouette: 0.5058
Diferencia absoluta: 0.1658
Posible sobreentrenamiento.
No hay subentrenamiento.
KMEANS - K=9
Train Silhouette: 0.3212
Validation Silhouette: 0.4538
Diferencia absoluta: 0.1326
Posible sobreentrenamiento.
Posible subentrenamiento.


GMM

In [29]:
from pyspark.ml.clustering import GaussianMixture

# ==========================================
# GMM
# ==========================================

best_gmm_model = None
best_gmm_score = -1
best_gmm_k = None

for k in k_values:

    gmm = GaussianMixture(
        k=k,
        seed=42,
        featuresCol="scaledFeatures",
        predictionCol="prediction"
    )

    model = gmm.fit(train_df)

    # TRAIN
    train_pred = model.transform(train_df)

    train_silhouette = evaluator.evaluate(train_pred)

    # VALIDATION
    val_pred = model.transform(val_df)

    val_silhouette = evaluator.evaluate(val_pred)

    diff_pct = abs(train_silhouette - val_silhouette)

    print("=" * 50)
    print(f"GMM - K={k}")
    print(f"Train Silhouette: {train_silhouette:.4f}")
    print(f"Validation Silhouette: {val_silhouette:.4f}")
    print(f"Diferencia absoluta: {diff_pct:.4f}")

    # Verificación sobreentrenamiento
    if diff_pct < 0.03:
        print("El modelo NO está sobreentrenado.")
    else:
        print("Posible sobreentrenamiento.")

    # Verificación subentrenamiento (> 0.50 en telemetría SOC indica clusters relativamente bien definidos)
    if train_silhouette < 0.50 and val_silhouette < 0.50:
      print("Posible subentrenamiento.")
    else:
      print("No hay subentrenamiento.")

    # Mejor modelo
    if val_silhouette > best_gmm_score:
        best_gmm_score = val_silhouette
        best_gmm_model = model
        best_gmm_k = k

GMM - K=5
Train Silhouette: 0.1292
Validation Silhouette: 0.2981
Diferencia absoluta: 0.1689
Posible sobreentrenamiento.
Posible subentrenamiento.
GMM - K=9
Train Silhouette: 0.2181
Validation Silhouette: 0.2859
Diferencia absoluta: 0.0678
Posible sobreentrenamiento.
Posible subentrenamiento.


###4.6 Selección del mejor modelo

In [30]:
print("=" * 60)
print("MEJOR MODELO KMEANS")
print(f"K óptimo: {best_k}")
print(f"Validation Silhouette: {best_kmeans_score:.4f}")

print("=" * 60)
print("MEJOR MODELO GMM")
print(f"K óptimo: {best_gmm_k}")
print(f"Validation Silhouette: {best_gmm_score:.4f}")

# Selección
if best_kmeans_score > best_gmm_score:

    final_model = best_kmeans_model
    final_model_name = "KMeans"

else:

    final_model = best_gmm_model
    final_model_name = "GMM"

print("=" * 60)
print(f"MEJOR MODELO FINAL: {final_model_name}")

MEJOR MODELO KMEANS
K óptimo: 5
Validation Silhouette: 0.5058
MEJOR MODELO GMM
K óptimo: 5
Validation Silhouette: 0.2981
MEJOR MODELO FINAL: KMeans


###4.7 Evaluación final en TEST

Luego del proceso de entrenamiento y validación de los modelos de aprendizaje no supervisado, se determinó que el mejor algoritmo fue K-Means, obteniendo:

- K=5
- Validation Silhouette Score = 0.5058

In [31]:
# ==========================================
# Evaluación final
# ==========================================

# Predicciones sobre TEST
test_pred = final_model.transform(test_df)

# Silhouette final
test_silhouette = evaluator.evaluate(test_pred)

print("=" * 60)
print(f"Modelo final: {final_model_name}")
print(f"Silhouette Score TEST: {test_silhouette:.4f}")

Modelo final: KMeans
Silhouette Score TEST: 0.3317


**Comentarios sobre los resultados obtenidos**

En esta actividad se implementaron algoritmos de aprendizaje no supervisado utilizando PySpark MLlib sobre un dataset masivo de ciberseguridad correspondiente al entorno SOC de Microsoft GUIDE. El objetivo principal fue identificar patrones latentes de comportamiento y evaluar la capacidad de agrupamiento de distintos algoritmos de clustering sobre telemetría de incidentes de seguridad.

**Preparación y preprocesamiento de datos**

El dataset original presentó características propias de ambientes reales de ciberseguridad:

- Alta cardinalidad.
- Gran volumen de registros.
- Múltiples variables categóricas.
- Fuerte presencia de datos faltantes.
- Desbalance entre categorías operativas.

Para abordar estas dificultades se realizaron varias etapas de preparación:

- Eliminación de registros sin variable objetivo.
- Eliminación de duplicados.
- Descarte de variables con más de 90% de valores NULL.
- Imputación de variables moderadamente incompletas.
- Generación de variables temporales.
- Reducción de cardinalidad.
- Construcción de muestras estratificadas representativas.

La estrategia de muestreo estratificado utilizando: IncidentGrade, CategoryGroup, EntityTypeGroup, EvidenceRole; permitió preservar adecuadamente la estructura operacional del dataset original dentro de la muestra M y posteriormente en la submuestra M’.

**División de entrenamiento, validación y prueba**

Inicialmente se consideró una división aleatoria simple, pero posteriormente se adoptó una estrategia más adecuada basada en OrgId, separando organizaciones completas entre: entrenamiento, validación y prueba.

Esta decisión fue metodológicamente correcta para un problema de ciberseguridad, ya que evitó:
- Leakage entre conjuntos.
- Sobreestimación artificial de desempeño.
- Dependencia entre eventos pertenecientes a una misma organización.

Además, esta técnica permitió evaluar mejor la capacidad de generalización de los modelos frente a organizaciones no observadas durante el entrenamiento.

**Selección de variables y transformación**

Debido a la alta cardinalidad de las variables categóricas, se utilizó Frequency Encoding en lugar de One-Hot Encoding.

Esta decisión fue adecuada considerando el tamaño del dataset, la necesidad de escalabilidad y la compatibilidad con algoritmos basados en distancia como K-Means y GMM.

Posteriormente se aplicó VectorAssembler y StandardScaler para construir y normalizar el vector de características.

El escalamiento fue especialmente importante porque K-Means y GMM dependen directamente de distancias euclidianas y distribuciones geométricas.

**Evaluación del uso de PCA**

Posteriormente se incorporó PCA (Principal Component Analysis) con el objetivo de reducir dimensionalidad, eliminar redundancia y mejorar eficiencia computacional.

Sin embargo, al aplicar PCA se observó un deterioro en las métricas de clustering, por lo que finalmente se decidió retirarlo.

Este comportamiento puede explicarse porque:
- Las variables utilizadas ya eran relativamente compactas.k
- Frequency Encoding reducía significativamente la dimensionalidad.
- PCA probablemente eliminó información relevante para distinguir patrones de comportamiento en ciberseguridad.

**Evaluación de K-Means y GMM**

Se entrenaron dos algoritmos utilizando distintos valores de k:

- K-Means
- Gaussian Mixture Model (GMM)

La métrica seleccionada fue el Silhouette Score porque es una de las métricas más apropiadas para clustering no supervisado.

Los resultados mostraron que:

| Modelo        | Mejor Validation Score |
| ------------- | ---------------------- |
| K-Means (k=5) | 0.5058                 |
| GMM (k=5)     | 0.2981                 |

K-Means obtuvo un desempeño considerablemente superior al de GMM. Posibles razones del bajo rendimiento de GMM:
- El dataset presenta alta complejidad y ruido.
- Existen distribuciones muy desbalanceadas.
- El tamaño de muestra puede no haber sido suficiente para estimar adecuadamente matrices de covarianza complejas.

**Sobreentrenamiento y subentrenamiento**

Los modelos mostraron diferencias importantes entre entrenamiento y validación.

Por ejemplo:
| Modelo     | Diferencia |
| ---------- | ---------- |
| KMeans k=5 | 0.1658     |
| GMM k=5    | 0.1689     |

La diferencia fue mayor al umbral definido del 3%, por lo que se concluyó que existía posible sobreentrenamiento.

Sin embargo, en clustering no supervisado esta interpretación debe hacerse con cautela, porque:
- No existen etiquetas reales.
- Las distribuciones pueden variar naturalmente entre organizaciones.
- El split por OrgId genera diferencias estructurales legítimas entre conjuntos.

Por ello, parte de la diferencia observada probablemente refleja verdadera heterogeneidad organizacional, más que únicamente sobreajuste clásico.

También se observaron indicadores de posible subentrenamiento en algunos modelos, especialmente en GMM, debido a valores bajos de Silhouette Score.

**Evaluación final**

| Modelo final | Silhouette TEST |
| ------------ | --------------- |
| K-Means      | 0.3317          |

Un valor cercano a 0.33 puede considerarse razonable para un problema real de ciberseguridad altamente complejo y ruidoso.

En datasets SOC reales los comportamientos suelen solaparse,
los ataques evolucionan constantemente y existen múltiples eventos benignos similares a amenazas.

Por ello, obtener clusters moderadamente separados representa un resultado aceptable y coherente con la naturaleza del problema.

En general, la actividad permitió aplicar exitosamente técnicas de aprendizaje no supervisado sobre Big Data de ciberseguridad utilizando PySpark, siguiendo una metodología consistente de preparación, entrenamiento, validación y evaluación de modelos.

**Repositorio del proyecto:** [GitHub - Actividad 4](https://github.com/cpalma-dev/BigData/blob/main/Actividad4_A01686081.ipynb)
